In [21]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

In [22]:
df = pd.read_csv(
    "./f_nearest_neighbors.csv",
    index_col=0
)

In [23]:
df = pd.read_csv("./new_fff_ppp.csv",index_col=0)

In [24]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
Interpolated,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Interpolated,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Interpolated,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Interpolated,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Interpolated,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# %%bash
# head ./avg_realbiopsies.csv

In [26]:
# remove where 9 is a child of 0

In [27]:
trees = [
#     {
# 	18: [19],
# 	19: [],
# 	7: [18, 17],
# 	17: [],
# 	4: [7],
# 	3: [15, 14],
# 	15: [],
# 	14: [],
# 	2: [13],
# 	13: [],
# 	1: [9],
# 	9: [],
# 	0: [16, 12, 11, 10, 8, 6, 5, 4, 3, 2, 1],
# 	16: [],
# 	12: [],
# 	11: [],
# 	10: [],
# 	8: [],
# 	6: [],
# 	5: []
# }, 
    {
	18: [19],
	19: [],
	7: [18, 17, 16],
	17: [],
	16: [],
	4: [7],
	3: [15, 14],
	15: [],
	14: [],
	2: [13],
	13: [],
	1: [9],
	9: [],
	0: [12, 11, 10, 8, 6, 5, 4, 3, 2, 1],
	12: [],
	11: [],
	10: [],
	8: [],
	6: [],
	5: []
}, 
#     {
# 	18: [19],
# 	19: [],
# 	7: [18, 17],
# 	17: [],
# 	4: [7],
# 	3: [15, 14],
# 	15: [],
# 	14: [],
# 	2: [13],
# 	13: [],
# 	1: [9, 8],
# 	9: [],
# 	8: [],
# 	0: [16, 12, 11, 10, 6, 5, 4, 3, 2, 1],
# 	16: [],
# 	12: [],
# 	11: [],
# 	10: [],
# 	6: [],
# 	5: []
# }, 
    # original below
    {
	18: [19],
	19: [],
	7: [18, 17, 16],
	17: [],
	16: [],
	4: [7],
	3: [15, 14],
	15: [],
	14: [],
	2: [13],
	13: [],
	1: [9, 8],
	9: [],
	8: [],
	0: [12, 11, 10, 6, 5, 4, 3, 2, 1],
	12: [],
	11: [],
	10: [],
	6: [],
	5: []
}, ]

In [28]:
m = len(df)
n = len(df.columns)

f_hat = df.to_numpy(dtype=float)
f = np.zeros((m, n))

# Variables: [f_0, ..., f_n-1, g_0, ..., g_n-1]

# Objective function
c = np.zeros(2 * n)
c[0] = n - 1
c[n+1:] = 1

for p in range(m):
    A_ub = []
    b_ub = []

    # All variables must be between 0 and 1
    bounds = [(0, 1)] * (2 * n)

    # Root frequency constraint
    if max(f_hat[p]) >= 0.01:
        bounds[0] = (0.7, 1)

    for i in range(n):

        if i > 0:
            # g_i >= f_i - f_hat_i
            row = np.zeros(2 * n)
            row[i] = 1
            row[n + i] = -1

            A_ub.append(row)
            b_ub.append(f_hat[p, i])

            # g_i >= f_hat_i - f_i
            row = np.zeros(2 * n)
            row[i] = -1
            row[n + i] = -1

            A_ub.append(row)
            b_ub.append(-f_hat[p, i])

        # Phylogenetic constraints
        for delta in trees:
            row = np.zeros(2 * n)
            row[i] = -1

            for j in delta[i]:
                row[j] = 1

            A_ub.append(row)
            b_ub.append(0)

    result = linprog(
        c,
        A_ub=np.array(A_ub),
        b_ub=np.array(b_ub),
        bounds=bounds,
        method="highs"
    )

    if not result.success:
        raise RuntimeError(
            f"Optimization failed at row {p}: {result.message}"
        )

    f[p] = result.x[:n]

print("Optimization complete!")
print(f.shape)

Optimization complete!
(1848, 20)


In [30]:
np.savetxt(
    "f_nearest_neighbors_lp.csv",
    f,
    delimiter=","
)